In [1]:
!pip install -q -U bitsandbytes
import json
import re
import time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

CFG_TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

MODEL_NAME = "Qwen/Qwen3-8B"

# 4-bit quantization -- keeps the full 8B model within a T4's 16GB VRAM.
# Full bf16 weights alone would need ~16GB (8B params x 2 bytes), leaving no
# room for generation. 4-bit compresses that to ~4GB, real headroom left over.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {MODEL_NAME} in 4-bit... (first run downloads the weights, needs internet -- fine for this prep step)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=quantization_config, device_map="auto"
)
print("Model loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def load_reports(csv_path: str, report_column: str):
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=[report_column])
    return df


def extract_json_block(raw: str) -> str:
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        return match.group(0)
    raise ValueError("No JSON object found in response")


# ═══════════════════════════════════════════════════════════════════
# the prompt
# now asks for a graded severity (null/0/1/2) instead of binary presence
# (null/0/1)
# ═══════════════════════════════════════════════════════════════════
def build_extraction_prompt_graded(report_text: str) -> str:
    targets_list = ", ".join(CFG_TARGETS)
    prompt = (
        f"Analyze the following radiology report and grade each of these 12 findings: {targets_list}.\n"
        f"Report: {report_text}\n"
        "For each finding, use ONLY strict literal reading of the report text, and grade the "
        "SEVERITY of the language used, not just whether the term appears:\n"
        "- null if the report does not explicitly mention this specific finding\n"
        "- 0 if the report explicitly states this finding is absent or normal\n"
        "- 1 if the report mentions this finding using MILD/TRACE language "
        "(e.g. 'trace', 'small', 'minimal', 'slight')\n"
        "- 2 if the report mentions this finding using CLEAR/SIGNIFICANT language "
        "(e.g. 'moderate', 'large', 'significant', 'tear', 'rupture', with no mild/trace qualifier)\n"
        "Do not infer a finding is absent just because a similar or nearby structure was mentioned instead.\n"
        "Respond with ONLY the JSON object — no explanation, no notes, no markdown code fences, "
        "no text before or after the JSON."
    )
    return prompt


def extract_labels_for_report(report_text: str, max_retries: int = 3) -> dict:
    prompt = build_extraction_prompt_graded(report_text)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,  # Qwen3-specific: skip internal reasoning, go straight to the answer
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    for attempt in range(max_retries):
        try:
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs, max_new_tokens=300, do_sample=False,
                )
            new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
            raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            cleaned = extract_json_block(raw)
            return json.loads(cleaned)
        except (json.JSONDecodeError, Exception) as e:
            if attempt == max_retries - 1:
                print(f"Failed after {max_retries} attempts: {e}")
                print(f"RAW OUTPUT WAS: {repr(raw[:500])}")  # <- see exactly what the model actually produced
                return {t: None for t in CFG_TARGETS}
            time.sleep(1)
    return {t: None for t in CFG_TARGETS}


def run_extraction(df: pd.DataFrame, report_column: str, uid_column: str, output_path: str):
    
    results = {}
    for i, row in df.iterrows():
        uid = row[uid_column]
        report = row[report_column]
        labels = extract_labels_for_report(report)
        results[uid] = labels
        if i % 50 == 0:
            print(f"Processed {i}/{len(df)}")
            with open(output_path, "w") as f:
                json.dump(results, f, indent=2)
    with open(output_path, "w") as f:
        json.dump(results, f, indent=2)
    return results


def severity_to_binary(severity):
    """Collapses graded severity (null/0/1/2) down to binary (null/0/1) so it
    can be checked against existing binary hard labels. Matches the
    label side of severity_to_label_and_weight() in
    label_quality_improvements.py -- both severity 1 (mild) and 2 (clear)
    count as "present" for agreement purposes, since the hard labels don't
    carry a severity distinction to compare against."""
    if severity is None:
        return None
    if severity == 0:
        return 0
    elif severity in (1, 2):
        return 1
    else:
        raise ValueError(f"Unexpected severity value: {severity}")


def validate_against_hard_labels_graded(extracted: dict, hard_labeled_df: pd.DataFrame, uid_column: str):
    """Same comparison as validate_against_hard_labels() in the binary
    script, but first collapses this run's severity grades to binary via
    severity_to_binary() so it lines up with the hard-labeled 0/1 columns.
    A separate severity breakdown is also printed, so you can see whether
    disagreements cluster at severity 1 (mild/trace language) -- the
    documented, expected soft spot -- rather than at severity 2."""
    hard_labeled_df = hard_labeled_df.set_index(uid_column)

    # Binary comparison (for a like-for-like number against your original
    # ~0.78 binary-extraction agreement)
    extracted_binary = {
        uid: {t: severity_to_binary(sev) for t, sev in severities.items()}
        for uid, severities in extracted.items()
    }
    extracted_df = pd.DataFrame.from_dict(extracted_binary, orient="index")
    extracted_df = extracted_df.reindex(hard_labeled_df.index)
    comparison_df = hard_labeled_df.join(extracted_df, lsuffix="_hard", rsuffix="_extracted")

    agreement = {}
    counts = {}
    for target in CFG_TARGETS:
        hard_col = f"{target}_hard"
        extracted_col = f"{target}_extracted"
        valid_rows = comparison_df[extracted_col].notnull()
        matches = (comparison_df.loc[valid_rows, hard_col] == comparison_df.loc[valid_rows, extracted_col]).sum()
        total = valid_rows.sum()
        agreement[target] = matches / total if total > 0 else None
        counts[target] = total

    agreement_df = pd.DataFrame.from_dict(agreement, orient="index", columns=["agreement"])
    agreement_df["n"] = pd.Series(counts)
    print("Binary agreement (severity 1 or 2 both counted as 'present'):")
    print(agreement_df)
    overall_agreement = agreement_df["agreement"].mean()
    print(f"Overall binary agreement: {overall_agreement:.2f}")

    # Severity breakdown of the disagreements, so mild-language cases can be
    # told apart from clear-language cases in the mismatches.
    raw_extracted_df = pd.DataFrame.from_dict(extracted, orient="index").reindex(hard_labeled_df.index)
    raw_comparison_df = hard_labeled_df.join(raw_extracted_df, lsuffix="_hard", rsuffix="_severity")
    print("\nSeverity distribution among disagreements (hard label vs. binary-collapsed extraction differ):")
    for target in CFG_TARGETS:
        hard_col = f"{target}_hard"
        sev_col = f"{target}_severity"
        binary_col_mask = comparison_df[f"{target}_extracted"].notnull() & \
            (comparison_df[hard_col] != comparison_df[f"{target}_extracted"])
        mismatched_severities = raw_comparison_df.loc[binary_col_mask, sev_col]
        if len(mismatched_severities) > 0:
            print(f"  {target}: {mismatched_severities.value_counts().to_dict()} (n={len(mismatched_severities)} disagreements)")

    return overall_agreement


if __name__ == "__main__":
    REPORT_COLUMN = "Report"
    UID_COLUMN = "StudyInstanceUID"

    DATA_PATH = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
    full_df = load_reports(DATA_PATH, REPORT_COLUMN)
    hard_labeled = pd.read_csv(DATA_PATH).dropna(subset=CFG_TARGETS)

    # ── STEP A: Validation check -- run this once, first ──
    RUN_VALIDATION = False

    if RUN_VALIDATION:
        validation_subset = full_df[full_df[UID_COLUMN].isin(hard_labeled[UID_COLUMN])]
        print(f"Running validation pass on {len(validation_subset)} known-labeled studies first...")
        validation_extracted = run_extraction(validation_subset, REPORT_COLUMN, UID_COLUMN, "validation_check_graded.json")
        validate_against_hard_labels_graded(validation_extracted, hard_labeled, UID_COLUMN)
        print("\nReview the binary-collapsed agreement above against ~0.78 (your original binary-extraction run).")
        print("A close match means the graded prompt isn't degrading extraction quality just by asking a harder question.")
        print("If comparable, set RUN_VALIDATION = False and rerun to move on to full extraction below.")

    else:
        # ── STEP B: The full, multi-session extraction ──────────────────
        # Set this to None for a fresh first session; on session 2+, point it
        # at the previous session's outpu t JSON attached as a Kaggledataset.
        PRIOR_SESSION_OUTPUT = "/kaggle/input/datasets/chiragggg/extracted-labels-graded/extracted_labels_graded_this_session.json"
        OUTPUT_PATH = "/kaggle/working/extracted_labels_graded.json"
        REPORTS_PER_SESSION = 1600

        if PRIOR_SESSION_OUTPUT is not None:
            with open(PRIOR_SESSION_OUTPUT, "r") as f:
                already_done = json.load(f)
            print(f"Resuming from prior session: {len(already_done)} reports already extracted.")
        else:
            already_done = {}
            print("Starting fresh -- no prior session output provided.")

        remaining_df = full_df[~full_df[UID_COLUMN].isin(already_done.keys())]
        this_session_df = remaining_df.iloc[:REPORTS_PER_SESSION]  # only ever process this capped slice
        print(f"Remaining this session: {len(remaining_df)} of {len(full_df)} total reports.")
        print(f"Processing this session: {len(this_session_df)} (capped at {REPORTS_PER_SESSION})")

        if len(this_session_df) == 0:
            print("Nothing left to extract -- all reports already done in a prior session.")
        else:
            # NOTE: capped to this_session_df here, unlike the binary
            # notebook's STEP B which accidentally passed remaining_df (the
            # uncapped remainder) into run_extraction -- that would silently
            # ignore REPORTS_PER_SESSION and try to process everything left
            # in one sitting. Fixed here since the whole point of the cap is
            # to fit inside one Kaggle session's time limit.
            new_results = run_extraction(this_session_df, REPORT_COLUMN, UID_COLUMN, "extracted_labels_graded_this_session.json")
            combined = {**already_done, **new_results}
            with open(OUTPUT_PATH, "w") as f:
                json.dump(combined, f, indent=2)
            print(f"Saved combined total: {len(combined)} reports to {OUTPUT_PATH}")

        # Remember to COMMIT/SAVE this notebook version once it finishes or
        # hits the session limit, so the NEXT session can attach it as input.

    # ── STEP C: Once extraction is fully done (all ~4,349 reports covered
    # across sessions), convert straight to the label table your training
    # script's find_label_table() expects, with per-target confidence
    # weights baked in -- no separate conversion step needed later.
    #
    # from label_quality_improvements import convert_extraction_to_label_table
    # with open("/kaggle/working/extracted_labels_graded.json") as f:
    #     final_extracted = json.load(f)
    # convert_extraction_to_label_table(final_extracted, "/kaggle/working/report_labels_graded.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 41.5 MB/s eta 0:00:00


Loading Qwen/Qwen3-8B in 4-bit... (first run downloads the weights, needs internet -- fine for this prep step)


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded.
GPU memory allocated: 1.57 GB


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Resuming from prior session: 1600 reports already extracted.
Remaining this session: 2807 of 4407 total reports.
Processing this session: 1600 (capped at 1600)
Processed 1600/1600
Processed 1650/1600
Processed 1700/1600
Processed 1750/1600
Processed 1800/1600
Processed 1850/1600
Processed 1900/1600
Processed 1950/1600
Processed 2000/1600
Processed 2050/1600
Processed 2100/1600
Processed 2150/1600
Processed 2200/1600
Processed 2250/1600
Processed 2300/1600
Processed 2350/1600
Processed 2400/1600
Processed 2450/1600
Processed 2500/1600
Processed 2550/1600
Processed 2600/1600
Processed 2650/1600
Processed 2700/1600
Processed 2750/1600
Processed 2800/1600
Processed 2850/1600
Processed 2900/1600
Processed 2950/1600
Processed 3000/1600
Processed 3050/1600
Processed 3100/1600
Processed 3150/1600
Saved combined total: 3200 reports to /kaggle/working/extracted_labels_graded.json
